# Experiment: Traffic Flow Prophet (Colab T4)

Objective:
- Build a daily 7-day park visitor forecast pipeline from the PRD.
- Use Prophet with China holidays and weather regressors.
- Produce `ds` + non-negative `yhat` output that can be used by dashboard/backend jobs.


## Colab Runtime Notes

- In Colab, set runtime to **GPU (T4)** if available: `Runtime -> Change runtime type -> T4 GPU`.
- Prophet itself is CPU-oriented, but this notebook is fully compatible with a T4 Colab runtime.
- Run cells top-to-bottom.


In [ ]:
# Runtime check (works in Colab and local Jupyter)
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    gpu_info = subprocess.check_output(["nvidia-smi"], text=True)
    print("GPU detected:\n")
    print(gpu_info.splitlines()[0])
except Exception:
    print("No GPU visible. This notebook still runs on CPU.")


In [ ]:
# Install dependencies in Colab (safe to rerun)
%pip -q install prophet pandas numpy matplotlib requests


In [ ]:
from __future__ import annotations

import datetime as dt
import os
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from prophet import Prophet

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1) Load Historical Data

Expected columns (PRD):
- `ds`: date
- `y`: real visitor count
- `temp_max`: max daily temperature
- `weather_score`: numeric weather penalty coefficient

If `/content/historical_flow.csv` is missing, this notebook creates mock 90-day data so the full pipeline can run.


In [ ]:
REQUIRED_COLUMNS = ["ds", "y", "temp_max", "weather_score"]
MULTI_FEATURE_COLUMNS = ["precip", "wind_speed_day", "humidity", "uv_index", "is_rain", "is_severe_weather"]


def validate_training_df(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    out = df.copy()
    out["ds"] = pd.to_datetime(out["ds"])
    out["y"] = pd.to_numeric(out["y"])
    out["temp_max"] = pd.to_numeric(out["temp_max"])
    out["weather_score"] = pd.to_numeric(out["weather_score"])
    for col in MULTI_FEATURE_COLUMNS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.sort_values("ds").reset_index(drop=True)
    return out


def enrich_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "precip" not in out.columns:
        out["precip"] = np.select(
            [out["weather_score"] <= 0.3, out["weather_score"] <= 0.6, out["weather_score"] <= 0.85],
            [15.0, 5.0, 1.0],
            default=0.0,
        )
    out["precip"] = out["precip"].fillna(0.0).clip(lower=0)

    if "wind_speed_day" not in out.columns:
        out["wind_speed_day"] = 3.0 + (1 - out["weather_score"]).clip(lower=0) * 6
    out["wind_speed_day"] = out["wind_speed_day"].fillna(out["wind_speed_day"].median()).clip(lower=0)

    if "humidity" not in out.columns:
        out["humidity"] = 60 + (1 - out["weather_score"]).clip(lower=0) * 25
    out["humidity"] = out["humidity"].fillna(65).clip(lower=10, upper=100)

    if "uv_index" not in out.columns:
        out["uv_index"] = (out["temp_max"] - 10) / 2
    out["uv_index"] = out["uv_index"].fillna(out["uv_index"].median()).clip(lower=0, upper=12)

    out["is_rain"] = ((out["precip"] > 0.0) | (out["weather_score"] < 0.75)).astype(int)
    out["is_severe_weather"] = ((out["precip"] >= 10.0) | (out["weather_score"] <= 0.35)).astype(int)
    return out


def build_mock_historical_data(days: int = 90) -> pd.DataFrame:
    end_date = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    ds = pd.date_range(end=end_date, periods=days, freq="D")
    day_of_week = ds.dayofweek
    weekend_boost = np.where(day_of_week >= 5, 1800, 0)
    base = 3200 + weekend_boost

    temp_max = 20 + 8 * np.sin(np.linspace(0, 3 * np.pi, days)) + np.random.normal(0, 1.8, days)
    temp_max = np.clip(temp_max, 5, 38)

    precip = np.random.choice([0.0, 0.3, 1.5, 5.0, 12.0], size=days, p=[0.55, 0.15, 0.15, 0.1, 0.05])
    wind_speed_day = np.random.uniform(2.0, 8.0, size=days)
    humidity = np.random.uniform(45, 90, size=days)
    uv_index = np.clip((temp_max - 8) / 2 + np.random.normal(0, 0.8, days), 0, 11)

    weather_score = 1.0 - 0.015 * precip - 0.01 * np.maximum(wind_speed_day - 4, 0) - 0.004 * np.maximum(humidity - 75, 0) - 0.008 * np.maximum(uv_index - 8, 0)
    weather_score = np.clip(weather_score, 0.2, 1.0)
    is_rain = (precip > 0.0).astype(int)
    is_severe_weather = ((precip >= 10.0) | (weather_score <= 0.35)).astype(int)

    noise = np.random.normal(0, 220, days)
    y = base * weather_score + (temp_max - 20) * 55 - precip * 50 - is_severe_weather * 600 + noise
    y = np.clip(y, 0, None).round().astype(int)

    return pd.DataFrame(
        {
            "ds": ds,
            "y": y,
            "temp_max": temp_max.round(1),
            "weather_score": weather_score.round(3),
            "precip": precip.round(2),
            "wind_speed_day": wind_speed_day.round(1),
            "humidity": humidity.round(1),
            "uv_index": uv_index.round(1),
            "is_rain": is_rain,
            "is_severe_weather": is_severe_weather,
        }
    )


HIST_PATH = "/content/historical_flow.csv"
if os.path.exists(HIST_PATH):
    df_train = pd.read_csv(HIST_PATH)
    source = f"Loaded from {HIST_PATH}"
else:
    df_train = build_mock_historical_data(days=90)
    source = "Generated mock historical data (file not found)"

df_train = validate_training_df(df_train)
df_train = enrich_weather_features(df_train)
print(source)
print(f"Rows: {len(df_train)} | Date range: {df_train['ds'].min().date()} -> {df_train['ds'].max().date()}")
df_train.head()


## 2) Build, Compare, and Train Prophet Models

We train two candidates and select the better one by holdout validation (MAE + MAPE):
- Baseline model: regressors `temp_max`, `weather_score`
- Multi-weather model: `temp_max`, `precip`, `wind_speed_day`, `humidity`, `uv_index`, `is_rain`, `is_severe_weather`


In [ ]:
BASELINE_REGRESSORS = ["temp_max", "weather_score"]
MULTI_REGRESSORS = ["temp_max", "precip", "wind_speed_day", "humidity", "uv_index", "is_rain", "is_severe_weather"]


def build_prophet_model(regressors: list[str]) -> Prophet:
    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=False,
    )
    model.add_country_holidays(country_name="CN")
    for col in regressors:
        prior_scale = 0.2 if col == "temp_max" else 0.1
        model.add_regressor(col, prior_scale=prior_scale)
    return model


def fit_prophet(df: pd.DataFrame, regressors: list[str]) -> Prophet:
    model = build_prophet_model(regressors)
    model.fit(df[["ds", "y"] + regressors].copy())
    return model


def evaluate_regressor_set(df: pd.DataFrame, regressors: list[str], holdout_days: int = 14) -> dict:
    holdout_days = max(7, min(holdout_days, len(df) // 4))
    if len(df) < 45 or holdout_days < 7:
        return {"mae": np.nan, "mape": np.nan, "holdout_days": 0}

    train_df = df.iloc[:-holdout_days].copy()
    valid_df = df.iloc[-holdout_days:].copy()

    model = fit_prophet(train_df, regressors)
    pred = model.predict(valid_df[["ds"] + regressors])[["ds", "yhat"]]

    merged = valid_df[["ds", "y"]].merge(pred, on="ds", how="left")
    abs_err = (merged["y"] - merged["yhat"]).abs()
    mae = float(abs_err.mean())
    mape = float((abs_err / merged["y"].clip(lower=1)).mean() * 100)
    return {"mae": mae, "mape": mape, "holdout_days": int(holdout_days)}


model_catalog = {
    "baseline_weather_score": BASELINE_REGRESSORS,
    "multi_weather_regressors": MULTI_REGRESSORS,
}

comparison_rows = []
for name, regressors in model_catalog.items():
    missing = [c for c in regressors if c not in df_train.columns]
    if missing:
        comparison_rows.append({"model": name, "regressors": ",".join(regressors), "mae": np.nan, "mape": np.nan, "holdout_days": 0, "status": f"missing columns: {missing}"})
        continue

    metrics = evaluate_regressor_set(df_train, regressors, holdout_days=14)
    status = "evaluated" if metrics["holdout_days"] > 0 else "trained_without_holdout"
    comparison_rows.append({
        "model": name,
        "regressors": ",".join(regressors),
        "mae": metrics["mae"],
        "mape": metrics["mape"],
        "holdout_days": metrics["holdout_days"],
        "status": status,
    })

model_comparison = pd.DataFrame(comparison_rows)
scored = model_comparison.dropna(subset=["mae"]).sort_values("mae")
if not scored.empty:
    selected_model_name = str(scored.iloc[0]["model"])
else:
    selected_model_name = "baseline_weather_score"

selected_regressors = model_catalog[selected_model_name]
model = fit_prophet(df_train, selected_regressors)

print(f"Selected model: {selected_model_name}")
print("Regressors:", selected_regressors)
model_comparison


## 2.1) Interpretability: Regressor Coefficients

This section prints coefficient direction and relative magnitude for the selected Prophet model regressors.


In [ ]:
try:
    from prophet.utilities import regressor_coefficients

    coef_df = regressor_coefficients(model).copy()
    coef_df = coef_df[coef_df['regressor'].isin(selected_regressors)].copy()

    if coef_df.empty:
        print('No regressor coefficients available for the selected model.')
    else:
        coef_df['abs_coef'] = coef_df['coef'].abs()
        coef_df = coef_df.sort_values('abs_coef', ascending=False).reset_index(drop=True)
        print(f'Selected model: {selected_model_name}')
        print('Interpretation: positive coef increases forecast, negative coef decreases forecast (all else equal).')
        display(coef_df[['regressor', 'coef', 'coef_lower', 'coef_upper', 'abs_coef']])
except Exception as e:
    print('Could not extract regressor coefficients from Prophet utilities.')
    print('Reason:', e)


## 3) Build Future 7-Day Features

This cell calls your QWeather 7-day forecast API and builds a composite weather score from weather text, precipitation, wind, humidity, UV, and temperature comfort. If unavailable, it falls back to safe default values.


In [ ]:
@dataclass
class WeatherConfig:
    endpoint: str = "https://ne63ywenn9.re.qweatherapi.com/v7/weather/7d"
    location: str = "113.892186,22.533620"  # lon,lat
    api_key: str = os.getenv("QWEATHER_API_KEY", "8b6d317a0a1e4a31812a625907dc6b39")


def _to_float(v, default: float = 0.0) -> float:
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


def qweather_row_to_score(row: dict) -> float:
    text_day = str(row.get("textDay", ""))
    text_night = str(row.get("textNight", ""))
    text = f"{text_day}/{text_night}"
    score = 1.0

    severe_weather = [
        "暴雨", "大暴雨", "特大暴雨", "暴雪", "强对流", "台风", "龙卷", "冰雹", "沙尘暴", "冻雨", "雷暴",
    ]
    medium_weather = ["大雨", "中雨", "中雪", "大雪", "雨夹雪", "雷阵雨", "扬沙", "浮尘"]
    mild_weather = ["小雨", "阵雨", "小雪", "阴", "雾", "霾"]

    if any(k in text for k in severe_weather):
        score -= 0.45
    elif any(k in text for k in medium_weather):
        score -= 0.28
    elif any(k in text for k in mild_weather):
        score -= 0.12
    elif "多云" in text:
        score -= 0.04

    precip = _to_float(row.get("precip"), 0.0)
    if precip >= 20:
        score -= 0.35
    elif precip >= 10:
        score -= 0.25
    elif precip >= 3:
        score -= 0.15
    elif precip > 0:
        score -= 0.06

    wind_speed = _to_float(row.get("windSpeedDay"), 0.0)
    if wind_speed >= 12:
        score -= 0.20
    elif wind_speed >= 8:
        score -= 0.12
    elif wind_speed >= 5:
        score -= 0.05

    humidity = _to_float(row.get("humidity"), 60.0)
    if humidity >= 90 or humidity <= 25:
        score -= 0.08
    elif humidity >= 80 or humidity <= 35:
        score -= 0.04

    uv_index = _to_float(row.get("uvIndex"), 0.0)
    if uv_index >= 10:
        score -= 0.10
    elif uv_index >= 8:
        score -= 0.06
    elif uv_index >= 6:
        score -= 0.03

    temp_max = _to_float(row.get("tempMax"), 22.0)
    temp_min = _to_float(row.get("tempMin"), 16.0)
    mean_temp = (temp_max + temp_min) / 2
    if mean_temp <= 5 or mean_temp >= 35:
        score -= 0.20
    elif mean_temp <= 10 or mean_temp >= 32:
        score -= 0.12
    elif mean_temp <= 15 or mean_temp >= 30:
        score -= 0.06

    return float(np.clip(score, 0.2, 1.0))


def fetch_future_weather(config: WeatherConfig, days: int = 7) -> pd.DataFrame:
    if not config.api_key:
        raise ValueError("Missing QWeather API key. Set QWEATHER_API_KEY env var.")

    params = {
        "location": config.location,
        "key": config.api_key,
    }
    response = requests.get(config.endpoint, params=params, timeout=20)
    response.raise_for_status()
    payload = response.json()

    if payload.get("code") != "200":
        raise ValueError(f"QWeather error: {payload}")

    daily = payload.get("daily", [])[:days]
    if len(daily) < days:
        raise ValueError(f"Expected {days} daily rows, got {len(daily)}")

    out = pd.DataFrame(
        {
            "ds": pd.to_datetime([d["fxDate"] for d in daily]),
            "temp_max": pd.to_numeric([d["tempMax"] for d in daily]),
            "weather_score": [qweather_row_to_score(d) for d in daily],
            "weather_text_day": [d.get("textDay", "") for d in daily],
            "weather_text_night": [d.get("textNight", "") for d in daily],
            "precip": pd.to_numeric([d.get("precip", 0) for d in daily]),
            "wind_speed_day": pd.to_numeric([d.get("windSpeedDay", 0) for d in daily]),
            "humidity": pd.to_numeric([d.get("humidity", 0) for d in daily]),
            "uv_index": pd.to_numeric([d.get("uvIndex", 0) for d in daily]),
        }
    )
    out["is_rain"] = (out["precip"] > 0.0).astype(int)
    out["is_severe_weather"] = ((out["precip"] >= 10.0) | (out["weather_score"] <= 0.35)).astype(int)
    out["weather_score"] = out["weather_score"].round(3)
    return out.sort_values("ds").reset_index(drop=True)


def fallback_future_features(days: int = 7) -> pd.DataFrame:
    ds = pd.date_range(start=pd.Timestamp.today().normalize() + pd.Timedelta(days=1), periods=days, freq="D")
    temp_max = np.linspace(22, 27, days)
    weather_score = np.array([0.96, 0.93, 0.86, 0.72, 0.55, 0.88, 0.92])[:days]
    precip = np.array([0.0, 0.0, 0.2, 2.0, 8.0, 0.5, 0.0])[:days]
    wind_speed_day = np.array([3.0, 3.0, 4.0, 6.0, 8.0, 4.0, 3.0])[:days]
    humidity = np.array([62, 65, 70, 78, 85, 72, 68])[:days]
    uv_index = np.array([7, 8, 7, 6, 5, 7, 8])[:days]

    out = pd.DataFrame({
        "ds": ds,
        "temp_max": temp_max,
        "weather_score": weather_score,
        "precip": precip,
        "wind_speed_day": wind_speed_day,
        "humidity": humidity,
        "uv_index": uv_index,
    })
    out["is_rain"] = (out["precip"] > 0.0).astype(int)
    out["is_severe_weather"] = ((out["precip"] >= 10.0) | (out["weather_score"] <= 0.35)).astype(int)
    return out


cfg = WeatherConfig()
try:
    future_df = fetch_future_weather(cfg, days=7)
    print("Fetched future weather from QWeather API.")
except Exception as e:
    print(f"Weather API failed ({e}). Using fallback defaults.")
    future_df = fallback_future_features(days=7)

future_df


## 4) Predict Next 7 Days and Post-process

Use the selected model from step 2. Important PRD rule: clip negative predictions to `0` before downstream usage.


In [ ]:
missing_future_cols = [c for c in selected_regressors if c not in future_df.columns]
if missing_future_cols:
    raise ValueError(f"Future weather features missing columns required by selected model: {missing_future_cols}")

future_for_model = future_df[["ds"] + selected_regressors].copy()
forecast = model.predict(future_for_model)

results = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
results["yhat"] = results["yhat"].clip(lower=0).round().astype(int)
results["yhat_lower"] = results["yhat_lower"].clip(lower=0).round().astype(int)
results["yhat_upper"] = results["yhat_upper"].clip(lower=0).round().astype(int)
results["model_name"] = selected_model_name
results


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_train["ds"], df_train["y"], label="history", alpha=0.5)
plt.plot(results["ds"], results["yhat"], marker="o", label="forecast")
plt.fill_between(results["ds"], results["yhat_lower"], results["yhat_upper"], alpha=0.2, label="interval")
plt.title("Park Visitor Forecast (7 Days)")
plt.xlabel("Date")
plt.ylabel("Visitors")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()


## 5) Export for Backend / Dashboard Job

This mirrors PRD output behavior and writes an idempotent-friendly daily result file.


In [ ]:
OUT_PATH = "/content/flow_prediction_daily.csv"
results[["ds", "yhat"]].to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")
results[["ds", "yhat"]]


## Production Notes

- Put your API key in env var `QWEATHER_API_KEY` to avoid hardcoding credentials.
- Prefer multi-regressor weather features (`precip`, `wind_speed_day`, `humidity`, `uv_index`, weather flags) when available.
- If weather API fails in production, backfill weather features with safe defaults and compute `is_rain` / `is_severe_weather` deterministically.
- Keep a manual override layer in backend for sudden closures/events.
- Daily scheduling can run this as a Python job (for example, via crontab).
